<a href="https://colab.research.google.com/github/Yogi-Puvvala/patient_readmission/blob/main/Patient_Readmission.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pickle

In [2]:
with open("X_train_num.pkl", "rb") as f:
    X_train_num = pickle.load(f)

In [3]:
with open("X_test_num.pkl", "rb") as f:
    X_test_num = pickle.load(f)

In [6]:
with open("X_train_cat.pkl", "rb") as f:
    X_train_cat = pickle.load(f)

In [7]:
with open("X_test_cat.pkl", "rb") as f:
    X_test_cat = pickle.load(f)

In [8]:
with open("X_train_text.pkl", "rb") as f:
    X_train_text = pickle.load(f)

In [9]:
with open("X_test_text.pkl", "rb") as f:
    X_test_text = pickle.load(f)

In [10]:
with open("X_train_seq.pkl", "rb") as f:
    X_train_seq = pickle.load(f)

In [11]:
with open("X_test_seq.pkl", "rb") as f:
    X_test_seq = pickle.load(f)

In [12]:
with open("y_train.pkl", "rb") as f:
    y_train = pickle.load(f)

In [13]:
with open("y_test.pkl", "rb") as f:
    y_test = pickle.load(f)

In [17]:
import tensorflow
from tensorflow import keras
from keras.layers import *
from keras.models import Model, Sequential

In [27]:
# Params
max_text_length = 3
vocab_size      = 20
embedding_dim   = 8

# Numerical Model
input_num  = Input(shape=(X_train_num.shape[1],), name="numerical")
x1         = Dense(16, activation="relu")(input_num)
x1         = Dropout(0.3)(x1)
x1         = Dense(8, activation="relu")(x1)

# Categorical Model
input_cat  = Input(shape=(X_train_cat.shape[1],), name="categorical")
x2         = Dense(16, activation="relu")(input_cat)
x2         = Dropout(0.3)(x2)
x2         = Dense(8, activation="relu")(x2)

# Textual Model
input_text = Input(shape=(max_text_length,), name="text")
x3         = Embedding(input_dim=vocab_size,
                       output_dim=embedding_dim,
                       input_length=max_text_length)(input_text)
x3         = LSTM(16, return_sequences=True)(x3)
x3         = LSTM(8)(x3)

# Sequential Model
input_seq  = Input(shape=(5, 1), name="sequential")
x4         = LSTM(16, return_sequences=True)(input_seq)
x4         = LSTM(8)(x4)

# Combining all branches
combined   = Concatenate()([x1, x2, x3, x4])
x          = Dense(16, activation="relu")(combined)
x          = Dropout(0.3)(x)
output     = Dense(3, activation="softmax", name="output")(x)

# Final Model
model = Model(
    inputs  = [input_num, input_cat, input_text, input_seq],
    outputs = output
)

In [28]:
model.compile(
    optimizer = "adam",
    loss      = "sparse_categorical_crossentropy",  # integer labels
    metrics   = ["accuracy"]
)

In [29]:
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ numerical           │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ categorical         │ (None, 42)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ text (InputLayer)   │ (None, 3)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 16)        │         80 │ numerical[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 16)        │        688 │ categorical[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, 3, 8)      │        160 │ text[0][0]        │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ sequential          │ (None, 5, 1)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_6 (Dropout) │ (None, 16)        │          0 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 16)        │          0 │ dense_12[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_8 (LSTM)       │ (None, 3, 16)     │      1,600 │ embedding_2[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_10 (LSTM)      │ (None, 5, 16)     │      1,152 │ sequential[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 8)         │        136 │ dropout_6[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 8)         │        136 │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_9 (LSTM)       │ (None, 8)         │        800 │ lstm_8[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_11 (LSTM)      │ (None, 8)         │        800 │ lstm_10[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_2       │ (None, 32)        │          0 │ dense_11[0][0],   │
│ (Concatenate)       │                   │            │ dense_13[0][0],   │
│                     │                   │            │ lstm_9[0][0],     │
│                     │                   │            │ lstm_11[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 16)        │        528 │ concatenate_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 16)        │          0 │ dense_14[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ output (Dense)      │ (None, 3)         │         51 │ dropout_8[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 6,131 (23.95 KB)

 Trainable params: 6,131 (23.95 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
history = model.fit(
    x = {
        "numerical"  : X_train_num,    # shape: (n, 4)
        "categorical": X_train_cat,    # shape: (n, 4)
        "text"       : X_train_text,   # shape: (n, 10)
        "sequential" : X_train_seq     # shape: (n, 5, 1)
    },

    y = y_train,

    epochs          = 20,
    batch_size      = 32,

    validation_data = (
        {
            "numerical"  : X_test_num,
            "categorical": X_test_cat,
            "text"       : X_test_text,
            "sequential" : X_test_seq
        },
        y_test
    ),

    callbacks = [
        tensorflow.keras.callbacks.EarlyStopping(
            monitor              = "val_loss",
            patience             = 5,
            restore_best_weights = True
        ),
        tensorflow.keras.callbacks.ReduceLROnPlateau(
            monitor  = "val_loss",
            factor   = 0.5,
            patience = 3
        )
    ],

    verbose = 1
)

Epoch 1/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 43s 14ms/step - accuracy: 0.5316 - loss: 0.9479 - val_accuracy: 0.5424 - val_loss: 0.9190 - learning_rate: 0.0010
Epoch 2/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - accuracy: 0.5374 - loss: 0.9245 - val_accuracy: 0.5424 - val_loss: 0.9151 - learning_rate: 0.0010
Epoch 3/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 42s 14ms/step - accuracy: 0.5384 - loss: 0.9199 - val_accuracy: 0.5423 - val_loss: 0.9129 - learning_rate: 0.0010
Epoch 4/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - accuracy: 0.5415 - loss: 0.9180 - val_accuracy: 0.5498 - val_loss: 0.9106 - learning_rate: 0.0010
Epoch 5/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 33s 14ms/step - accuracy: 0.5435 - loss: 0.9159 - val_accuracy: 0.5494 - val_loss: 0.9103 - learning_rate: 0.0010
Epoch 6/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 32s 13ms/step - accuracy: 0.5435 - loss: 0.9162 - val_accuracy: 0.5515 - val_loss: 0.9085 - learning_rate: 0.0010
Epoch 7/20
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 33s 14ms/step - accura

In [38]:
from google.colab import drive
drive.mount('/content/drive')

# Save model
import pickle

with open('/content/drive/MyDrive/model.pkl', 'wb') as f:
    pickle.dump(model, f)

Mounted at /content/drive
